In [ ]:
import duckdb
import json
import requests

In [ ]:
def fetch_pdf(arxiv_id: str, output_dir: Path) -> Path:
    """Download a paper's PDF. Returns the local path on success."""
    url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    pdf_path = output_dir / f"{arxiv_id}.pdf"

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(pdf_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

    return pdf_path

In [ ]:
def fetch_source(arxiv_id: str, output_dir: Path) -> Path:
    """Download a paper's LaTeX source tarball."""
    url = f"https://arxiv.org/e-print/{arxiv_id}"
    latex_path = output_dir / f"{arxiv_id}.tar.gz"

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(latex_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

    return latex_path

In [ ]:
def update_paper_paths(conn: duckdb.DuckDBPyConnection, arxiv_id: str, pdf_path: Path, source_path: Path) -> None:
    """Mark a paper as fetched in the papers table."""
    conn.execute("""
                 UPDATE papers
                 SET pdf_path = ?, latex_source_path = ?
                 WHERE arxiv_id = ?
                 """,
                 [str(pdf_path), str(source_path), arxiv_id],

)    

In [ ]:
def get_papers_to_fetch(conn: duckdb.DuckDBPyConnection) -> list[str]:
    """Return arxiv_ids from benchmark_subset whose pdf_path is still NULL in papers."""
    rows = conn.execute("""
        SELECT bs.arxiv_id
        FROM benchmark_subset bs
        JOIN papers p USING (arxiv_id)
        WHERE p.pdf_path IS NULL
        ORDER BY bs.arxiv_id
    """).fetchall()
    return [row[0] for row in rows]

In [ ]:
def main():
    """Orchestrator: loop over papers, fetch, update DB, sleep, retry."""
    conn = duckdb.connect('data/arxiv_metadata.duckdb')
    os.makedirs("data/raw/pdfs/", exist_ok=True) 
    os.makedirs("data/raw/sources/", exist_ok=True) 
    arxiv_ids = get_papers_to_fetch(conn)
    for id in arxiv_ids:
        fetch_pdf(id)
        fetch_source(id)
        update_paper_paths(conn, id)
        time.sleep(3)  # Be polite to arXiv's servers
        print(f"Fetched {id}")
    conn.close()